<img src="https://raw.githubusercontent.com/Vidhusv/FRAMEXPLORER/main/framexplore_logo.png" height="200" align="right" style="height:240px">

## FrameXplore v1.0: Disulfide bond correction for cysteine-rich protein structures

Automatically identify and correct disulfide bonds in protein structures predicted by [ColabFold](https://github.com/sokrypton/ColabFold) / [AlphaFold2](https://www.nature.com/articles/s41586-021-03819-2). Built on top of ColabFold v1.6.1, FrameXplore adds a one‑click **post‑prediction analysis** that finds true cysteine pairs using geometric thresholds, rewrites the PDB with proper `CONECT` records, and visualizes the corrected disulfide network — no extra input required.

For detailed instructions, see the <a href="#Instructions">bottom</a> of the notebook.  
For the source code, latest updates, and citation information, visit the [FrameXplore GitHub](https://github.com/Vidhusv/FRAMEXPLORER) and the [official Zenodo record](https://doi.org/10.5281/zenodo.20132827).

**Citation**  
> Vijay, V. S. (2026). *FrameXplore: Automated disulfide bond correction for cysteine-rich protein structures* (v1.0.0). Zenodo. https://doi.org/10.5281/zenodo.20132827

**ColabFold reference**  
> Mirdita M, Schütze K, Moriwaki Y, Heo L, Ovchinnikov S, Steinegger M. ColabFold: Making protein folding accessible to all. *Nature Methods*, 2022. https://doi.org/10.1038/s41592-022-01488-1



In [ ]:
#@title Input protein sequence(s), then hit `Runtime` -> `Run all`
from google.colab import files
import os
import re
import hashlib
import random

from sys import version_info
python_version = f"{version_info.major}.{version_info.minor}"

def add_hash(x,y):
  return x+"_"+hashlib.sha1(y.encode()).hexdigest()[:5]

query_sequence = 'MGMRMMFTVFLLVVLATTVVSFPSERASDGRDDTAKDEGSDMDKLVEKKECCNPACGRHYSCGR' #@param {type:"string"}
#@markdown  - Use `:` to specify inter-protein chainbreaks for **modeling complexes** (supports homo- and hetro-oligomers). For example **PI...SK:PI...SK** for a homodimer
jobname = 'alpha con' #@param {type:"string"}
# number of models to use
num_relax = 0 #@param [0, 1, 5] {type:"raw"}
#@markdown - specify how many of the top ranked structures to relax using amber
template_mode = "none" #@param ["none", "pdb100","custom"]
#@markdown - `none` = no template information is used. `pdb100` = detect templates in pdb100 (see [notes](#pdb100)). `custom` - upload and search own templates (PDB or mmCIF format, see [notes](#custom_templates))

use_amber = num_relax > 0

# remove whitespaces
query_sequence = "".join(query_sequence.split())

basejobname = "".join(jobname.split())
basejobname = re.sub(r'\W+', '', basejobname)
jobname = add_hash(basejobname, query_sequence)

# check if directory with jobname exists
def check(folder):
  if os.path.exists(folder):
    return False
  else:
    return True
if not check(jobname):
  n = 0
  while not check(f"{jobname}_{n}"): n += 1
  jobname = f"{jobname}_{n}"

# make directory to save results
os.makedirs(jobname, exist_ok=True)

# save queries
queries_path = os.path.join(jobname, f"{jobname}.csv")
with open(queries_path, "w") as text_file:
  text_file.write(f"id,sequence\n{jobname},{query_sequence}")

if template_mode == "pdb100":
  use_templates = True
  custom_template_path = None
elif template_mode == "custom":
  custom_template_path = os.path.join(jobname,f"template")
  os.makedirs(custom_template_path, exist_ok=True)
  uploaded = files.upload()
  use_templates = True
  for fn in uploaded.keys():
    os.rename(fn,os.path.join(custom_template_path,fn))
else:
  custom_template_path = None
  use_templates = False

print("jobname",jobname)
print("sequence",query_sequence)
print("length",len(query_sequence.replace(":","")))

In [ ]:
#@title Install dependencies
%%time
import os
USE_AMBER = use_amber
USE_TEMPLATES = use_templates
PYTHON_VERSION = python_version

if not os.path.isfile("COLABFOLD_READY"):
  print("installing colabfold...")
  os.system("pip install -q --no-warn-conflicts 'colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold'")
  if os.environ.get('TPU_NAME', False) != False:
    os.system("pip uninstall -y jax jaxlib")
    os.system("pip install --no-warn-conflicts --upgrade dm-haiku==0.0.10 'jax[cuda12_pip]'==0.3.25 -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html")
  os.system("ln -s /usr/local/lib/python3.*/dist-packages/colabfold colabfold")
  os.system("ln -s /usr/local/lib/python3.*/dist-packages/alphafold alphafold")
  # hack to fix TF crash
  os.system("rm -f /usr/local/lib/python3.*/dist-packages/tensorflow/core/kernels/libtfkernel_sobol_op.so /usr/local/lib/python3.*/dist-packages/tensorflow/lite/python/*/*.so")
  os.system("touch COLABFOLD_READY")

if USE_AMBER or USE_TEMPLATES:
  if not os.path.isfile("CONDA_READY"):
    print("installing conda...")
    os.system("wget -qnc https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh")
    os.system("bash Miniforge3-Linux-x86_64.sh -bfp /usr/local")
    os.system("mamba config --set auto_update_conda false")
    os.system("touch CONDA_READY")

if USE_TEMPLATES and not os.path.isfile("HH_READY") and USE_AMBER and not os.path.isfile("AMBER_READY"):
  print("installing hhsuite and amber...")
  os.system(f"mamba install -y -c conda-forge -c bioconda kalign2=2.04 hhsuite=3.3.0 openmm=8.2.0 python='{PYTHON_VERSION}' pdbfixer")
  os.system("touch HH_READY")
  os.system("touch AMBER_READY")
else:
  if USE_TEMPLATES and not os.path.isfile("HH_READY"):
    print("installing hhsuite...")
    os.system(f"mamba install -y -c conda-forge -c bioconda kalign2=2.04 hhsuite=3.3.0 python='{PYTHON_VERSION}'")
    os.system("touch HH_READY")
  if USE_AMBER and not os.path.isfile("AMBER_READY"):
    print("installing amber...")
    os.system(f"mamba install -y -c conda-forge openmm=8.2.0 python='{PYTHON_VERSION}' pdbfixer")
    os.system("touch AMBER_READY")

In [ ]:
#@markdown ### MSA options (custom MSA upload, single sequence, pairing mode)
msa_mode = "mmseqs2_uniref_env" #@param ["mmseqs2_uniref_env", "mmseqs2_uniref","single_sequence","custom"]
pair_mode = "unpaired_paired" #@param ["unpaired_paired","paired","unpaired"] {type:"string"}
#@markdown - "unpaired_paired" = pair sequences from same species + unpaired MSA, "unpaired" = seperate MSA for each chain, "paired" - only use paired sequences.

# decide which a3m to use
if "mmseqs2" in msa_mode:
  a3m_file = os.path.join(jobname,f"{jobname}.a3m")

elif msa_mode == "custom":
  a3m_file = os.path.join(jobname,f"{jobname}.custom.a3m")
  if not os.path.isfile(a3m_file):
    custom_msa_dict = files.upload()
    custom_msa = list(custom_msa_dict.keys())[0]
    header = 0
    import fileinput
    for line in fileinput.FileInput(custom_msa,inplace=1):
      if line.startswith(">"):
         header = header + 1
      if not line.rstrip():
        continue
      if line.startswith(">") == False and header == 1:
         query_sequence = line.rstrip()
      print(line, end='')

    os.rename(custom_msa, a3m_file)
    queries_path=a3m_file
    print(f"moving {custom_msa} to {a3m_file}")

else:
  a3m_file = os.path.join(jobname,f"{jobname}.single_sequence.a3m")
  with open(a3m_file, "w") as text_file:
    text_file.write(">1\n%s" % query_sequence)

In [ ]:
#@markdown ### Advanced settings
model_type = "auto" #@param ["auto", "alphafold2_ptm", "alphafold2_multimer_v1", "alphafold2_multimer_v2", "alphafold2_multimer_v3", "deepfold_v1", "alphafold2"]
#@markdown - if `auto` selected, will use `alphafold2_ptm` for monomer prediction and `alphafold2_multimer_v3` for complex prediction.
#@markdown Any of the mode_types can be used (regardless if input is monomer or complex).
num_recycles = "3" #@param ["auto", "0", "1", "3", "6", "12", "24", "48"]
#@markdown - if `auto` selected, will use `num_recycles=20` if `model_type=alphafold2_multimer_v3`, else `num_recycles=3` .
recycle_early_stop_tolerance = "auto" #@param ["auto", "0.0", "0.5", "1.0"]
#@markdown - if `auto` selected, will use `tol=0.5` if `model_type=alphafold2_multimer_v3` else `tol=0.0`.
relax_max_iterations = 200 #@param [0, 200, 2000] {type:"raw"}
#@markdown - max amber relax iterations, `0` = unlimited (AlphaFold2 default, can take very long)
pairing_strategy = "greedy" #@param ["greedy", "complete"] {type:"string"}
#@markdown - `greedy` = pair any taxonomically matching subsets, `complete` = all sequences have to match in one line.
calc_extra_ptm = False #@param {type:"boolean"}
#@markdown - return pairwise chain iptm/actifptm

#@markdown #### Sample settings
#@markdown -  enable dropouts and increase number of seeds to sample predictions from uncertainty of the model.
#@markdown -  decrease `max_msa` to increase uncertainity
max_msa = "auto" #@param ["auto", "512:1024", "256:512", "64:128", "32:64", "16:32"]
num_seeds = 1 #@param [1,2,4,8,16] {type:"raw"}
use_dropout = False #@param {type:"boolean"}

num_recycles = None if num_recycles == "auto" else int(num_recycles)
recycle_early_stop_tolerance = None if recycle_early_stop_tolerance == "auto" else float(recycle_early_stop_tolerance)
if max_msa == "auto": max_msa = None

#@markdown #### Save settings
save_all = False #@param {type:"boolean"}
save_recycles = False #@param {type:"boolean"}
save_to_google_drive = False #@param {type:"boolean"}
#@markdown -  if the save_to_google_drive option was selected, the result zip will be uploaded to your Google Drive
dpi = 200 #@param {type:"integer"}
#@markdown - set dpi for image resolution

if save_to_google_drive:
  from pydrive2.drive import GoogleDrive
  from pydrive2.auth import GoogleAuth
  from google.colab import auth
  from oauth2client.client import GoogleCredentials
  auth.authenticate_user()
  gauth = GoogleAuth()
  gauth.credentials = GoogleCredentials.get_application_default()
  drive = GoogleDrive(gauth)
  print("You are logged into Google Drive and are good to go!")

#@markdown Don't forget to hit `Runtime` -> `Run all` after updating the form.

In [ ]:
#@title Run Prediction
display_images = True #@param {type:"boolean"}

import sys
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
from Bio import BiopythonDeprecationWarning
warnings.simplefilter(action='ignore', category=BiopythonDeprecationWarning)
from pathlib import Path
from colabfold.download import download_alphafold_params, default_data_dir
from colabfold.utils import setup_logging
from colabfold.batch import get_queries, run, set_model_type
from colabfold.plot import plot_msa_v2

import os
import numpy as np
try:
  K80_chk = os.popen('nvidia-smi | grep "Tesla K80" | wc -l').read()
except:
  K80_chk = "0"
  pass
if "1" in K80_chk:
  print("WARNING: found GPU Tesla K80: limited to total length < 1000")
  if "TF_FORCE_UNIFIED_MEMORY" in os.environ:
    del os.environ["TF_FORCE_UNIFIED_MEMORY"]
  if "XLA_PYTHON_CLIENT_MEM_FRACTION" in os.environ:
    del os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"]

from colabfold.colabfold import plot_protein
from pathlib import Path
import matplotlib.pyplot as plt

# For some reason we need that to get pdbfixer to import
if use_amber and f"/usr/local/lib/python{python_version}/site-packages/" not in sys.path:
    sys.path.insert(0, f"/usr/local/lib/python{python_version}/site-packages/")

def input_features_callback(input_features):
  if display_images:
    plot_msa_v2(input_features)
    plt.show()
    plt.close()

def prediction_callback(protein_obj, length,
                        prediction_result, input_features, mode):
  model_name, relaxed = mode
  if not relaxed:
    if display_images:
      fig = plot_protein(protein_obj, Ls=length, dpi=150)
      plt.show()
      plt.close()

result_dir = jobname
log_filename = os.path.join(jobname,"log.txt")
setup_logging(Path(log_filename))

queries, is_complex = get_queries(queries_path)
model_type = set_model_type(is_complex, model_type)

if "multimer" in model_type and max_msa is not None:
  use_cluster_profile = False
else:
  use_cluster_profile = True

download_alphafold_params(model_type, Path("."))
results = run(
    queries=queries,
    result_dir=result_dir,
    use_templates=use_templates,
    custom_template_path=custom_template_path,
    num_relax=num_relax,
    msa_mode=msa_mode,
    model_type=model_type,
    num_models=5,
    num_recycles=num_recycles,
    relax_max_iterations=relax_max_iterations,
    recycle_early_stop_tolerance=recycle_early_stop_tolerance,
    num_seeds=num_seeds,
    use_dropout=use_dropout,
    model_order=[1,2,3,4,5],
    is_complex=is_complex,
    data_dir=Path("."),
    keep_existing_results=False,
    rank_by="auto",
    pair_mode=pair_mode,
    pairing_strategy=pairing_strategy,
    stop_at_score=float(100),
    prediction_callback=prediction_callback,
    dpi=dpi,
    zip_results=False,
    save_all=save_all,
    max_msa=max_msa,
    use_cluster_profile=use_cluster_profile,
    input_features_callback=input_features_callback,
    save_recycles=save_recycles,
    user_agent="colabfold/google-colab-main",
    calc_extra_ptm=calc_extra_ptm,
)
results_zip = f"{jobname}.result.zip"
os.system(f"zip -r {results_zip} {jobname}")

In [ ]:
#@title Display 3D structure {run: "auto"}
import py3Dmol
import glob
import matplotlib.pyplot as plt
from colabfold.colabfold import plot_plddt_legend
from colabfold.colabfold import pymol_color_list, alphabet_list
rank_num = 1 #@param ["1", "2", "3", "4", "5"] {type:"raw"}
color = "lDDT" #@param ["chain", "lDDT", "rainbow"]
show_sidechains = False #@param {type:"boolean"}
show_mainchains = False #@param {type:"boolean"}

tag = results["rank"][0][rank_num - 1]
jobname_prefix = ".custom" if msa_mode == "custom" else ""
pdb_filename = f"{jobname}/{jobname}{jobname_prefix}_unrelaxed_{tag}.pdb"
pdb_file = glob.glob(pdb_filename)

def show_pdb(rank_num=1, show_sidechains=False, show_mainchains=False, color="lDDT"):
  model_name = f"rank_{rank_num}"
  view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js',)
  view.addModel(open(pdb_file[0],'r').read(),'pdb')

  if color == "lDDT":
    view.setStyle({'cartoon': {'colorscheme': {'prop':'b','gradient': 'roygb','min':50,'max':90}}})
  elif color == "rainbow":
    view.setStyle({'cartoon': {'color':'spectrum'}})
  elif color == "chain":
    chains = len(queries[0][1]) + 1 if is_complex else 1
    for n,chain,color in zip(range(chains),alphabet_list,pymol_color_list):
       view.setStyle({'chain':chain},{'cartoon': {'color':color}})

  if show_sidechains:
    BB = ['C','O','N']
    view.addStyle({'and':[{'resn':["GLY","PRO"],'invert':True},{'atom':BB,'invert':True}]},
                        {'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
    view.addStyle({'and':[{'resn':"GLY"},{'atom':'CA'}]},
                        {'sphere':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
    view.addStyle({'and':[{'resn':"PRO"},{'atom':['C','O'],'invert':True}]},
                        {'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
  if show_mainchains:
    BB = ['C','O','N','CA']
    view.addStyle({'atom':BB},{'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})

  view.zoomTo()
  return view

show_pdb(rank_num, show_sidechains, show_mainchains, color).show()
if color == "lDDT":
  plot_plddt_legend().show()

In [ ]:
#@title Plots {run: "auto"}
from IPython.display import display, HTML
import base64
from html import escape

# see: https://stackoverflow.com/a/53688522
def image_to_data_url(filename):
  ext = filename.split('.')[-1]
  prefix = f'data:image/{ext};base64,'
  with open(filename, 'rb') as f:
    img = f.read()
  return prefix + base64.b64encode(img).decode('utf-8')

pae = ""
pae_file = os.path.join(jobname,f"{jobname}{jobname_prefix}_pae.png")
if os.path.isfile(pae_file):
    pae = image_to_data_url(pae_file)
cov = image_to_data_url(os.path.join(jobname,f"{jobname}{jobname_prefix}_coverage.png"))
plddt = image_to_data_url(os.path.join(jobname,f"{jobname}{jobname_prefix}_plddt.png"))
display(HTML(f"""
<style>
  img {{
    float:left;
  }}
  .full {{
    max-width:100%;
  }}
  .half {{
    max-width:50%;
  }}
  @media (max-width:640px) {{
    .half {{
      max-width:100%;
    }}
  }}
</style>
<div style="max-width:90%; padding:2em;">
  <h1>Plots for {escape(jobname)}</h1>
  { '<!--' if pae == '' else '' }<img src="{pae}" class="full" />{ '-->' if pae == '' else '' }
  <img src="{cov}" class="half" />
  <img src="{plddt}" class="half" />
</div>
"""))

In [ ]:
#@title ToxinFrame – Disulfide Correction & Downloads
#@markdown This cell replaces the original zip download. It automatically
#@markdown 1. extracts cysteine coordinates from the best relaxed model,
#@markdown 2. identifies true disulfide bonds using geometric cutoffs,
#@markdown 3. writes a corrected PDB with proper CONECT records,
#@markdown 4. downloads the corrected PDB and the original result zip.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from Bio.PDB import PDBParser
import os
import shutil

# ----- 1. Locate the best relaxed model -----
# (Uses ColabFold’s output directory and ranking)
tag = results["rank"][0][0]  # rank 1
pdb_path = f"{jobname}/{jobname}_relaxed_{tag}.pdb"
if not os.path.isfile(pdb_path):
    # fallback to unrelaxed if relaxation was off
    pdb_path = f"{jobname}/{jobname}_unrelaxed_{tag}.pdb"
print(f"Analysing: {pdb_path}")

# ----- 2. Parse cysteine atoms -----
parser = PDBParser(QUIET=True)
structure = parser.get_structure("toxin", pdb_path)
cys_residues = []
for model in structure:
    for chain in model:
        for res in chain:
            if res.get_resname() == "CYS":
                ca = res["CA"].get_coord() if "CA" in res else None
                sg = res["SG"].get_coord() if "SG" in res else None
                cys_residues.append({
                    "chain": chain.get_id(),
                    "resid": res.get_id()[1],
                    "CA": ca,
                    "SG": sg
                })

if not cys_residues:
    print("No cysteine residues found. ToxinFrame only works for cysteine‑rich peptides.")
else:
    n = len(cys_residues)
    ca_coords = np.array([c["CA"] for c in cys_residues])
    sg_coords = np.array([c["SG"] for c in cys_residues])
    ca_dist = np.linalg.norm(ca_coords[:, np.newaxis] - ca_coords[np.newaxis, :], axis=-1)
    sg_dist = np.linalg.norm(sg_coords[:, np.newaxis] - sg_coords[np.newaxis, :], axis=-1)

    # ----- 3. Print distance heatmap -----
    labels = [f"{c['chain']}:{c['resid']}" for c in cys_residues]
    plt.figure(figsize=(max(8, n*0.5), max(6, n*0.45)))
    sns.heatmap(sg_dist, annot=True, fmt=".1f", xticklabels=labels, yticklabels=labels,
                cmap="coolwarm", cbar_kws={'label': 'SG–SG distance (Å)'})
    plt.title("Cysteine Sulfur–Sulfur Distance Matrix")
    plt.tight_layout()
    plt.show()

    # ----- 4. Identify disulfide bonds -----
    THRESHOLD_CA = 6.5   # Å
    THRESHOLD_SG = 2.5   # Å
    true_bonds = []
    for i in range(n):
        for j in range(i+1, n):
            if ca_dist[i][j] < THRESHOLD_CA and sg_dist[i][j] < THRESHOLD_SG:
                true_bonds.append((i, j, ca_dist[i][j], sg_dist[i][j]))
                print(f"✔ Disulfide bond: {labels[i]} – {labels[j]}  (Cα–Cα: {ca_dist[i][j]:.2f} Å, S–S: {sg_dist[i][j]:.2f} Å)")

    if true_bonds:
        print(f"\nFound {len(true_bonds)} probable disulfide bonds.")

        # ----- 5. Write corrected PDB -----
        corrected_pdb_path = f"{jobname}/{jobname}_ToxinFrame_corrected.pdb"
        shutil.copy(pdb_path, corrected_pdb_path)

        # Build CONECT records
        conect_lines = []
        for bond in true_bonds:
            i, j, _, _ = bond
            id1 = cys_residues[i]["resid"]
            id2 = cys_residues[j]["resid"]
            conect_lines.append(f"CONECT{id1:5d}{id2:5d}\n")

        # Remove any previous CONECT lines and insert ours before END/TER
        with open(corrected_pdb_path, "r") as f:
            original_lines = f.readlines()

        new_lines = []
        conect_inserted = False
        for line in original_lines:
            if line.startswith("CONECT"):   # skip old CONECTs
                continue
            if (line.startswith("END") or line.startswith("TER")) and not conect_inserted:
                new_lines.extend(conect_lines)
                conect_inserted = True
            new_lines.append(line)

        with open(corrected_pdb_path, "w") as f:
            f.writelines(new_lines)

        print(f"\nCorrected PDB saved: {corrected_pdb_path}")

        # ----- 6. Download corrected PDB (single file) -----
        files.download(corrected_pdb_path)
    else:
        print("No disulfide bonds found with the current geometric cutoffs.")

# ----- 7. Also keep the original result zip download -----
if msa_mode == "custom":
    print("Don't forget to cite your custom MSA generation method.")

files.download(f"{jobname}.result.zip")

if save_to_google_drive == True and drive:
    uploaded = drive.CreateFile({'title': f"{jobname}.result.zip"})
    uploaded.SetContentFile(f"{jobname}.result.zip")
    uploaded.Upload()
    print(f"Uploaded {jobname}.result.zip to Google Drive with ID {uploaded.get('id')}")

In [ ]:
#@title ToxinFrame – 3D view of corrected disulfides {run: "auto"}
import py3Dmol

# Check if the previous cell has been executed
required_vars = ['true_bonds', 'cys_residues', 'corrected_pdb_path']
missing = [v for v in required_vars if v not in globals()]

if missing:
    print("The ToxinFrame analysis cell has not been run yet (or produced no results).")
    print("Please run the previous cell first (the 'ToxinFrame – Disulfide Correction & Downloads' cell).")
elif not true_bonds:
    print("No disulfide bonds to display.")
else:
    with open(corrected_pdb_path, 'r') as f:
        pdb_string = f.read()

    view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js')
    view.addModel(pdb_string, 'pdb')

    # Cartoon coloured by pLDDT (same style as original)
    view.setStyle({'cartoon': {'colorscheme': {'prop':'b','gradient': 'roygb','min':50,'max':90}}})

    # Show cysteines as sticks
    view.addStyle({'resn': 'CYS'}, {'stick': {'colorscheme': 'yellowCarbon', 'radius': 0.2}})

    # Draw dashed lines for each true disulfide bond
    for bond in true_bonds:
        i, j, _, _ = bond
        view.addCylinder({
            'start': {'chain': cys_residues[i]['chain'], 'resi': cys_residues[i]['resid'], 'atom': 'SG'},
            'end':   {'chain': cys_residues[j]['chain'], 'resi': cys_residues[j]['resid'], 'atom': 'SG'},
            'radius': 0.15,
            'color': 'orange',
            'dashed': True
        })

    view.zoomTo()
    view.show()

# Instructions <a name="Instructions"></a>

## What is FrameXplore?

FrameXplore is a **Google Colab notebook** that extends [ColabFold](https://github.com/sokrypton/ColabFold) (v1.6.1) with **automatic disulfide‑bond identification and correction** for cysteine‑rich proteins.  
It is designed for toxin structural biologists, venom researchers, and anyone studying cysteine‑rich peptides (ICK motifs, conotoxins, defensins, etc.) who needs to verify that AlphaFold / ColabFold placed the disulfide bonds correctly.

**Key features**
- Runs the **full ColabFold prediction** pipeline (MMseqs2 MSA generation, AlphaFold2 or AlphaFold2‑multimer).  
- After the structure is built, **automatically** extracts all cysteine residues from the top‑ranked relaxed (or unrelaxed) model.  
- Computes pairwise **Cα–Cα** and **S–S distance** matrices.  
- Identifies **true disulfide bonds** using conservative geometric thresholds.  
- Writes a **corrected PDB file** with proper `CONECT` records.  
- Provides an **interactive 3D view** showing the corrected structure with cysteines as yellow sticks and disulfide bonds as orange dashed lines.  
- Entirely one‑click – no extra steps beyond `Runtime` → `Run all`.

---

## Quick start

1. **Paste your protein sequence(s)** into the `query_sequence` field at the top of the notebook.  
   - Use `:` to separate chains if you’re modelling a complex.  
   - Example (α‑Conotoxin GI from *Conus geographus*): `ECCNPACGRHYSC`

2. **Click `Runtime` → `Run all`** (or run the cells one by one in order).  
   The pipeline will go through several stages.  
   The currently running step is indicated by a stop‑sign icon next to the cell.

3. **Wait for the prediction to finish.**  
   Typical runtime is 5–15 minutes (shorter for small peptides).

4. **Review the automatic FrameXplore analysis** – a heatmap, a list of disulfide bonds, a corrected PDB file, and an interactive 3D viewer appear without any further input.

---

## Output files

After the run completes you will have:

### Standard ColabFold outputs (inside `jobname.result.zip`)
- PDB formatted structures ranked by pLDDT (and pTMscore for complexes)  
- Plots: pLDDT, predicted alignment error (PAE), MSA coverage  
- MSA in A3M format  
- Log file, `scores.json`, and BibTeX citations for all tools/databases used  

### FrameXplore‑specific outputs (auto‑downloaded)
- `jobname_FrameXplore_corrected.pdb` – the best model with corrected disulfide `CONECT` records  
- Cysteine distance heatmap (displayed inside the notebook)  
- Text summary listing each confirmed disulfide bond with its Cα–Cα and S–S distances  
- Interactive 3D view with corrected disulfides highlighted

---

## How disulfide bond identification works

FrameXplore uses two purely geometric criteria that are highly conserved across all known disulfide bonds in the Protein Data Bank (PDB):

| Criterion  | Cutoff |
|------------|--------|
| Cα–Cα distance | < 6.5 Å |
| S–S (sulfur–sulfur) distance | < 2.5 Å |

A cysteine pair is considered a **true disulfide bond** only if **both** distances fall below the cutoffs.  
This method is structure‑based, reliable, and requires no machine learning or sequence‑pattern matching.

**Why is this needed?**  
AlphaFold2 and ColabFold often predict **incorrect disulfide connectivity** for cysteine‑rich venom peptides, sometimes even returning the wrong fold. FrameXplore provides an immediate, human‑readable validation and fixes the bonds in the final PDB.

---

## Interpreting the FrameXplore output

- **Heatmap**: Shows the S–S distance between every pair of cysteines. True bonds will show a small value (typically ~2.0 Å). Large numbers mean the pair is not bonded.  
- **Text output**: Lists each confirmed bond with exact distances, e.g.,  
  `✔ Disulfide bond: A:51 – A:56 (Cα–Cα: 6.23 Å, S–S: 1.84 Å)`  
- **3D view**: The protein cartoon is coloured by pLDDT (same scale as ColabFold). Cysteine side chains are shown as yellow sticks, and true disulfide bonds are drawn as orange dashed cylinders.

---

## Advanced usage

- **Complexes**: Use the usual ColabFold syntax (e.g., `SEQ1:SEQ2`) to predict oligomers. FrameXplore will analyse all cysteines across all chains.  
- **Custom MSA / templates**: All ColabFold advanced options (`msa_mode`, `template_mode`, etc.) remain fully functional.  
- **Relaxation**: FrameXplore preferentially uses the **relaxed** model if `use_amber` is enabled; otherwise it falls back to the unrelaxed rank‑1 model.

---

## Important notes

- FrameXplore relies on the geometric quality of the predicted structure. If the model is very low confidence (pLDDT < 50), disulfide bond assignments may be less reliable. Always inspect the pLDDT plot first.  
- This tool **does not** retrain or modify AlphaFold; it only post‑processes the output. The underlying structure prediction is identical to a standard ColabFold run.  
- For sequences with many cysteines, the analysis still completes in a few seconds after the prediction finishes.

---

## License & attribution

FrameXplore is built upon **ColabFold**, which is:  

- ColabFold source code: [MIT License](https://github.com/sokrypton/ColabFold/blob/main/LICENSE)  
- AlphaFold2 parameters: [CC BY-NC 4.0](https://creativecommons.org/licenses/by-sa/4.0/) (DeepMind)  

FrameXplore itself is released under the **MIT License**.

If you use FrameXplore in your research, please cite **both** the ColabFold paper and FrameXplore:

1. Mirdita, M. et al. ColabFold: making protein folding accessible to all.
Nature Methods, 2022. https://doi.org/10.1038/s41592-022-01488-1
2. Vijay, V. S. (2026). FrameXplore: Automated disulfide bond correction
for cysteine-rich protein structures (v1.0.0). Zenodo.
https://doi.org/10.5281/zenodo.20132827

## Bugs & suggestions

Please report issues for FrameXplore at:
https://github.com/Vidhusv/FRAMEXPLORER

For problems specific to the underlying ColabFold pipeline, see the [original ColabFold issue tracker](https://github.com/sokrypton/ColabFold/issues).

## Acknowledgements

- **ColabFold team** (Sergey Ovchinnikov, Milot Mirdita, Martin Steinegger) for the fantastic prediction notebook.  
- AlphaFold team (DeepMind) for open‑sourcing the model.  
- David Koes for the py3Dmol plugin that powers our 3D visualization.  
- Do‑Yoon Kim for the ColabFold logo.  
- KOBIC and Söding Lab for the MMseqs2 MSA server.  

*FrameXplore is a community extension to make disulfide‑rich protein structure validation faster and more reliable. Enjoy!*